<a href="https://colab.research.google.com/github/johnsonnoah2000/Buffalo-311-Data-Pipeline/blob/main/Buffalo_311_Pipeline_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# CELL 1 — Install & Import Libraries
!pip install anthropic -q   # -q means "quiet" — less output clutter

import pandas as pd
import numpy as np
import anthropic
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

print("✅ All libraries imported successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 635.9/635.9 kB 20.0 MB/s eta 0:00:00
✅ All libraries imported successfully!


In [6]:
from google.colab import files
uploaded = files.upload()

print("✅ File uploaded!")

Saving 311_Service_Requests_(July_2008_-__May_2024)_20260420.csv.csv to 311_Service_Requests_(July_2008_-__May_2024)_20260420.csv.csv
✅ File uploaded!


In [8]:
def ingest_data(filepath: str) -> pd.DataFrame:
    """Load the raw Buffalo 311 CSV into a DataFrame."""
    df = pd.read_csv(
        filepath,
        skipinitialspace=True,  # strips spaces after commas
        on_bad_lines="warn"     # skips corrupt rows instead of crashing
    )
    # Clean up column names: strip spaces, lowercase, replace spaces with underscores
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

    print(f"✅ Loaded {len(df):,} rows and {len(df.columns)} columns")
    print(f"   Columns found: {list(df.columns)}")
    return df

df_raw = ingest_data("311_Service_Requests_(July_2008_-__May_2024)_20260420.csv.csv")

# Preview the first 5 rows so you can see what the raw data looks like
print("\n--- First 5 rows of RAW data ---")
df_raw.head()

/tmp/ipykernel_9818/2971343333.py:3: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


✅ Loaded 1,129,274 rows and 34 columns
   Columns found: ['case_reference', 'open_date', 'closed_date', 'status', 'subject', 'reason', 'type', 'object_type', 'address_number', 'address_line_1', 'address_line_2', 'city', 'state', 'zipcode', 'property_id', 'location', 'latitude', 'longitude', 'council_district', 'council_district_2011', 'police_district', 'census_tract', 'census_block_group', 'census_block', 'neighborhood', 'x_coordinate', 'y_coordinate', '2010_census_tract', '2010_census_block_group', '2010_census_block', 'tractce20', 'geoid20_tract', 'geoid20_blockgroup', 'geoid20_block']

--- First 5 rows of RAW data ---


,case_reference,open_date,closed_date,status,subject,reason,type,object_type,address_number,address_line_1,...,neighborhood,x_coordinate,y_coordinate,2010_census_tract,2010_census_block_group,2010_census_block,tractce20,geoid20_tract,geoid20_blockgroup,geoid20_block
0,1002041241,04/22/2024 11:19:00 AM,04/24/2024 10:05:00 AM,Closed,Dept of Public Works,Sanitation,Pick and Pay (Req_Serv),Property,554,GOODYEAR,...,Genesee-Moselle,"-8,774,490.166","5,298,475.9659",36,3,3006,003600,36029003600,360290036002,360290036002001
1,1002000532,02/27/2024 03:20:00 PM,02/28/2024 07:04:00 AM,Closed,Dept of Public Works,Engineering - Street Repairs,Paving (Req_Serv),Street,NaN,TRACY,...,UNKNOWN,NaN,NaN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN
2,1002040292,04/17/2024 01:06:00 PM,04/24/2024 03:49:00 PM,Closed,Dept of Public Works,Sanitation,Totes Pickup (Req_Serv),Property,40,MONTANA,...,Genesee-Moselle,"-8,774,271.939","5,298,453.4154",36,3,3004,003600,36029003600,360290036003,360290036003004
3,1002041823,04/24/2024 11:10:00 AM,NaN,Open,Dept of Law,Freedom of Information,FOIL Records Police Dept (Req_Serv),Unknown,Unknown,Unknown,...,UNKNOWN,NaN,NaN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN
4,1002041820,04/24/2024 11:04:00 AM,04/26/2024 10:42:00 AM,Closed,Dept of Public Works,Sanitation,Totes Replace (Req_Serv),Property,351,ROESCH,...,Riverside,"-8,782,964.329","5,306,374.9589",58.01,1,1002,005801,36029005801,360290058011,360290058011002


In [9]:
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """Clean and standardize the Buffalo 311 dataset."""

    print(f"Before cleaning: {len(df):,} rows")

    # 1. Remove exact duplicate rows
    df = df.drop_duplicates()
    print(f"  After removing duplicates: {len(df):,} rows")

    # 2. Standardize text fields — consistent capitalization
    #    .str.strip() removes extra spaces
    #    .str.title() makes Every Word Capitalized
    #    .str.lower() makes everything lowercase
    if "type" in df.columns:
        df["type"] = df["type"].str.strip().str.title()
    if "status" in df.columns:
        df["status"] = df["status"].str.strip().str.lower()
    if "neighborhood" in df.columns:
        df["neighborhood"] = df["neighborhood"].str.strip().str.title()

    # 3. Parse dates — convert text strings into real datetime objects
    #    errors="coerce" turns bad dates into NaT (missing) instead of crashing
    if "open_date" in df.columns:
        df["open_date"] = pd.to_datetime(df["open_date"], errors="coerce")
    if "closed_date" in df.columns:
        df["closed_date"] = pd.to_datetime(df["closed_date"], errors="coerce")

    # 4. Fill missing neighborhoods with "Unknown"
    #    We keep the row — we just label the missing location
    if "neighborhood" in df.columns:
        df["neighborhood"] = df["neighborhood"].fillna("Unknown")

    # 5. Drop rows with no open_date — we can't analyze without it
    if "open_date" in df.columns:
        df = df.dropna(subset=["open_date"])

    # 6. Drop rows with no type — we can't classify without it
    if "type" in df.columns:
        df = df[df["type"].notna()]

    print(f"After cleaning: {len(df):,} rows")
    print(f"Total removed: {len(df_raw) - len(df):,} rows")
    return df

df_clean = clean_data(df_raw)

# Preview cleaned data
print("\n--- First 5 rows of CLEAN data ---")
df_clean.head()

Before cleaning: 1,129,274 rows
  After removing duplicates: 1,129,274 rows
After cleaning: 1,129,274 rows
Total removed: 0 rows

--- First 5 rows of CLEAN data ---


,case_reference,open_date,closed_date,status,subject,reason,type,object_type,address_number,address_line_1,...,neighborhood,x_coordinate,y_coordinate,2010_census_tract,2010_census_block_group,2010_census_block,tractce20,geoid20_tract,geoid20_blockgroup,geoid20_block
0,1002041241,2024-04-22 11:19:00,2024-04-24 10:05:00,closed,Dept of Public Works,Sanitation,Pick And Pay (Req_Serv),Property,554,GOODYEAR,...,Genesee-Moselle,"-8,774,490.166","5,298,475.9659",36,3,3006,003600,36029003600,360290036002,360290036002001
1,1002000532,2024-02-27 15:20:00,2024-02-28 07:04:00,closed,Dept of Public Works,Engineering - Street Repairs,Paving (Req_Serv),Street,NaN,TRACY,...,Unknown,NaN,NaN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN
2,1002040292,2024-04-17 13:06:00,2024-04-24 15:49:00,closed,Dept of Public Works,Sanitation,Totes Pickup (Req_Serv),Property,40,MONTANA,...,Genesee-Moselle,"-8,774,271.939","5,298,453.4154",36,3,3004,003600,36029003600,360290036003,360290036003004
3,1002041823,2024-04-24 11:10:00,NaT,open,Dept of Law,Freedom of Information,Foil Records Police Dept (Req_Serv),Unknown,Unknown,Unknown,...,Unknown,NaN,NaN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN
4,1002041820,2024-04-24 11:04:00,2024-04-26 10:42:00,closed,Dept of Public Works,Sanitation,Totes Replace (Req_Serv),Property,351,ROESCH,...,Riverside,"-8,782,964.329","5,306,374.9589",58.01,1,1002,005801,36029005801,360290058011,360290058011002


In [10]:
def transform(df: pd.DataFrame) -> pd.DataFrame:
    """Engineer new features from the cleaned 311 data."""

    # --- Time-based features ---
    # .dt accessor lets us pull date parts out of datetime columns
    df["open_month"] = df["open_date"].dt.month_name()       # "January", "February"...
    df["open_year"] = df["open_date"].dt.year                # 2022, 2023, 2024...
    df["open_day_of_week"] = df["open_date"].dt.day_name()   # "Monday", "Tuesday"...
    df["open_quarter"] = df["open_date"].dt.quarter          # 1, 2, 3, or 4

    # --- Resolution time ---
    # Subtract open date from closed date to get number of days
    # Result is NaN for requests that are still open
    if "closed_date" in df.columns:
        df["days_to_close"] = (df["closed_date"] - df["open_date"]).dt.days

        # Flag: did it take more than 30 days? True or False
        df["slow_response"] = df["days_to_close"] > 30

        # Flag: is the request still open? (no closed date = still open)
        df["still_open"] = df["closed_date"].isna()
    else:
        df["days_to_close"] = None
        df["slow_response"] = False
        df["still_open"] = True

    print("✅ New features created:")
    print("   open_month, open_year, open_day_of_week, open_quarter")
    print("   days_to_close, slow_response, still_open")
    return df

df_transformed = transform(df_clean)

# Preview with new columns
print("\n--- Sample with new features ---")
df_transformed[["type", "neighborhood", "open_month", "days_to_close", "slow_response", "still_open"]].head(10)


✅ New features created:
   open_month, open_year, open_day_of_week, open_quarter
   days_to_close, slow_response, still_open

--- Sample with new features ---


,type,neighborhood,open_month,days_to_close,slow_response,still_open
0,Pick And Pay (Req_Serv),Genesee-Moselle,April,1.0,False,False
1,Paving (Req_Serv),Unknown,February,0.0,False,False
2,Totes Pickup (Req_Serv),Genesee-Moselle,April,7.0,False,False
3,Foil Records Police Dept (Req_Serv),Unknown,April,NaN,False,True
4,Totes Replace (Req_Serv),Riverside,April,1.0,False,False
5,Open311 Housing,Riverside,May,NaN,False,True
6,Pick And Pay (Req_Serv),Upper West Side,May,0.0,False,False
7,Police Issue (Req_Serv),Fillmore-Leroy,February,1.0,False,False
8,Totes Combo (Req_Serv),University Heights,February,26.0,False,False
9,Pot Hole (Req_Serv),Seneca-Cazenovia,February,1.0,False,False


In [11]:
def classify_priority(row) -> str:
    """Assign a priority label to each 311 request."""

    # Check if the request is still open AND has been open a long time
    if row.get("still_open") == True:
        if pd.notna(row.get("open_date")):
            days_open = (pd.Timestamp.now() - row["open_date"]).days
            if days_open > 60:
                return "Critical — Long Overdue"

    # Check if the city was slow to respond
    if row.get("slow_response") == True:
        return "High — Slow Response"

    # Check if it's a high-priority infrastructure type
    infrastructure_types = [
        "Pothole", "Water Main Break", "Street Light",
        "Flooding", "Gas Leak", "Traffic Signal"
    ]
    if row.get("type") in infrastructure_types:
        return "High — Infrastructure"

    # Check if it's already resolved
    if row.get("status") == "closed":
        return "Resolved"

    # Default label if none of the above matched
    return "Standard"

df_transformed["priority"] = df_transformed.apply(classify_priority, axis=1)

print("✅ Rule-based classification complete!")
print("\n--- Priority label counts ---")
print(df_transformed["priority"].value_counts())

print("\n--- Sample classified records ---")
df_transformed[["type", "neighborhood", "days_to_close", "status", "priority"]].head(15)

✅ Rule-based classification complete!

--- Priority label counts ---
priority
Resolved                   1033639
High — Slow Response         89089
Critical — Long Overdue       6546
Name: count, dtype: int64

--- Sample classified records ---


,type,neighborhood,days_to_close,status,priority
0,Pick And Pay (Req_Serv),Genesee-Moselle,1.0,closed,Resolved
1,Paving (Req_Serv),Unknown,0.0,closed,Resolved
2,Totes Pickup (Req_Serv),Genesee-Moselle,7.0,closed,Resolved
3,Foil Records Police Dept (Req_Serv),Unknown,NaN,open,Critical — Long Overdue
4,Totes Replace (Req_Serv),Riverside,1.0,closed,Resolved
5,Open311 Housing,Riverside,NaN,open,Critical — Long Overdue
6,Pick And Pay (Req_Serv),Upper West Side,0.0,closed,Resolved
7,Police Issue (Req_Serv),Fillmore-Leroy,1.0,closed,Resolved
8,Totes Combo (Req_Serv),University Heights,26.0,closed,Resolved
9,Pot Hole (Req_Serv),Seneca-Cazenovia,1.0,closed,Resolved


In [12]:
from sklearn.preprocessing import LabelEncoder

# Step 1: Encode the 'type' column (text → numbers)
#         LabelEncoder converts: "Pothole" → 0, "Graffiti" → 1, etc.
le_type = LabelEncoder()
df_transformed["type_enc"] = le_type.fit_transform(
    df_transformed["type"].fillna("Unknown")
)

# Step 2: Build the feature matrix (X) and target labels (y)
#         X = the columns we feed into the model
#         y = the labels we want the model to predict
X = df_transformed[["type_enc", "days_to_close", "slow_response"]].copy()
X["slow_response"] = X["slow_response"].astype(int)  # convert True/False to 1/0
X = X.fillna(-1)  # ML models can't handle NaN — use -1 as "missing" signal

y = df_transformed["priority"]

# Step 3: Create and train the Random Forest model
#         n_estimators=100 means 100 decision trees
#         random_state=42 makes results reproducible
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X, y)

# Step 4: Generate predictions
df_transformed["predicted_priority"] = clf.predict(X)

print("✅ ML classification complete!")
print("\n--- Rule-based vs ML Predicted (first 15 rows) ---")
print(df_transformed[["type", "priority", "predicted_priority"]].head(15).to_string())

✅ ML classification complete!

--- Rule-based vs ML Predicted (first 15 rows) ---
                                   type                 priority       predicted_priority
0               Pick And Pay (Req_Serv)                 Resolved                 Resolved
1                     Paving (Req_Serv)                 Resolved                 Resolved
2               Totes Pickup (Req_Serv)                 Resolved                 Resolved
3   Foil Records Police Dept (Req_Serv)  Critical — Long Overdue  Critical — Long Overdue
4              Totes Replace (Req_Serv)                 Resolved                 Resolved
5                       Open311 Housing  Critical — Long Overdue  Critical — Long Overdue
6               Pick And Pay (Req_Serv)                 Resolved                 Resolved
7               Police Issue (Req_Serv)                 Resolved                 Resolved
8                Totes Combo (Req_Serv)                 Resolved                 Resolved
9                 

In [13]:
output_filename = "buffalo_311_pipeline_output.csv"

df_transformed.to_csv(output_filename, index=False)
# index=False means don't include the row numbers as a column

print(f"✅ Saved {len(df_transformed):,} records to {output_filename}")
print("\n--- Final output columns ---")
print(list(df_transformed.columns))

print("\n--- Final sample (first 25 rows) ---")
df_transformed.head()

# Download the file to your computer
files.download(output_filename)
print(f"\n✅ Download started for {output_filename}")

✅ Saved 1,129,274 records to buffalo_311_pipeline_output.csv

--- Final output columns ---
['case_reference', 'open_date', 'closed_date', 'status', 'subject', 'reason', 'type', 'object_type', 'address_number', 'address_line_1', 'address_line_2', 'city', 'state', 'zipcode', 'property_id', 'location', 'latitude', 'longitude', 'council_district', 'council_district_2011', 'police_district', 'census_tract', 'census_block_group', 'census_block', 'neighborhood', 'x_coordinate', 'y_coordinate', '2010_census_tract', '2010_census_block_group', '2010_census_block', 'tractce20', 'geoid20_tract', 'geoid20_blockgroup', 'geoid20_block', 'open_month', 'open_year', 'open_day_of_week', 'open_quarter', 'days_to_close', 'slow_response', 'still_open', 'priority', 'type_enc', 'predicted_priority']

--- Final sample (first 25 rows) ---


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Download started for buffalo_311_pipeline_output.csv


In [22]:
import google.colab.userdata
def build_data_summary() -> str:
    """
    Summarizes the fully processed dataset into text so Claude
    can understand and answer questions about it.
    Now includes priority labels and ML predictions since
    the full pipeline has already run by this point.
    """
    total_records      = len(df_transformed)
    still_open         = int(df_transformed["still_open"].sum())
    slow_responses     = int(df_transformed["slow_response"].sum())

    top_types          = df_transformed["type"].value_counts().head(10).to_string()
    top_hoods          = df_transformed["neighborhood"].value_counts().head(10).to_string()
    priority_breakdown = df_transformed["priority"].value_counts().to_string()
    ml_breakdown       = df_transformed["predicted_priority"].value_counts().to_string()
    avg_close          = (df_transformed
                          .groupby("type")["days_to_close"]
                          .mean().dropna()
                          .sort_values(ascending=False)
                          .head(10).round(1).to_string())
    hood_avg_close     = (df_transformed
                          .groupby("neighborhood")["days_to_close"]
                          .mean().dropna()
                          .sort_values(ascending=False)
                          .head(10).round(1).to_string())
    by_month           = df_transformed["open_month"].value_counts().to_string()
    by_year            = df_transformed["open_year"].value_counts().sort_index().to_string()
    by_day             = df_transformed["open_day_of_week"].value_counts().to_string()

    summary = f"""
    You are an expert data analyst for the City of Buffalo, NY.
    You have full knowledge of the Buffalo 311 Service Request dataset.
    This data has been fully cleaned, transformed, and classified —
    both by rule-based logic and a Random Forest ML model.
    Answer every question clearly and specifically using the data below.
    If you spot interesting patterns beyond what was asked, mention them.

    Total Records: {total_records:,}
    Still Open: {still_open:,}
    Slow Responses (closed > 30 days): {slow_responses:,}

    --- Top 10 Request Types ---
{top_types}

    --- Top 10 Neighborhoods ---
{top_hoods}

    --- Priority Breakdown (Rule-based) ---
{priority_breakdown}

    --- Priority Breakdown (ML Predicted) ---
{ml_breakdown}

    --- Top 10 Request Types by Average Days to Close ---
{avg_close}

    --- Top 10 Neighborhoods by Average Days to Close ---
{hood_avg_close}

    --- Requests by Open Month ---
{by_month}

    --- Requests by Open Year ---
{by_year}

    --- Requests by Open Day of Week ---
{by_day}
    """
    return summary
# Build the summary once — reused for every question
data_summary = build_data_summary()

# Stores the full conversation so Claude remembers context
conversation_history = []

# Initialize the Anthropic client (replace 'YOUR_API_KEY' with your actual key)
# Consider using Colab secrets for better security: https://colab.research.google.com/notebooks/snippets/secrets.ipynb
client = anthropic.Anthropic(api_key=google.colab.userdata.get('ANTHROPIC_API_KEY'))

def ask(question: str) -> str:
    """
    Ask Claude anything about the fully processed 311 dataset.
    Maintains conversation history so follow-up questions work naturally.
    """
    conversation_history.append({
        "role": "user",
        "content": question
    })

    msg = client.messages.create(
        model="claude-opus-4-5",
        max_tokens=500,
        system=data_summary,
        messages=conversation_history
    )

    response_text = msg.content[0].text

    conversation_history.append({
        "role": "assistant",
        "content": response_text
    })

    return response_text

def reset_conversation():
    """
    Clears conversation history to start a fresh topic.
    """
    conversation_history.clear()
    print("✅ Conversation reset — starting fresh!")

print("✅ AI Data Analyst is ready — full pipeline data loaded!")
print("   Use ask('your question') to query Claude.")
print("   Use reset_conversation() to start a new topic.\n")

✅ AI Data Analyst is ready — full pipeline data loaded!
   Use ask('your question') to query Claude.
   Use reset_conversation() to start a new topic.



In [ ]:
ask("Ask your Question here")